# Streaming Multi Node


Streaming Across Multiple Nodes [Step 05 - Multi-Node Progress Tracking]

> **MLCourse - Agentic AI - LangGraph**

Real LangGraph applications have many nodes chained together. This notebook
builds a three-node pipeline (preprocess -> process -> postprocess) and shows
how `stream()` tracks progress as execution flows through each node. We also
demonstrate `astream_events` across multiple nodes for token-level visibility.

# What you will learn

1. Building a multi-node sequential graph.
2. How `stream()` yields one dict per node in execution order.
3. Using `stream_mode` options for different output formats.
4. Tracking progress through a multi-step pipeline.

### Key takeaways

- Each `stream()` yield corresponds to one node completing.
- You can count yields to build a progress bar.
- `stream_mode="updates"` gives you only the state diffs per node.

### Setup: imports, environment


In [ ]:
import os                           # env access
from dotenv import load_dotenv      # .env loading

load_dotenv(override=False)         # load without overriding

from typing import Annotated, TypedDict  # typed state

from langgraph.graph import StateGraph, END  # graph primitives
from langchain_ollama import ChatOllama      # local LLM
from langchain_core.messages import HumanMessage  # message type


### Model guard: check Ollama availability


In [ ]:
try:                                        # connectivity test
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("Ollama reachable")
except Exception as exc:
    LLM_AVAILABLE = False
    print("Ollama not reachable:", exc)


### Define pipeline state: carries data through all three nodes


In [ ]:
class PipelineState(TypedDict):
    """State tracks the data as it flows through preprocess -> process -> postprocess."""
    raw_input: str                         # original user input
    preprocessed: str                      # output of preprocessing step
    processed: str                         # output of main processing step
    final_output: str                      # output after postprocessing


### Node 1: preprocess -- clean and normalize the input


In [ ]:
def preprocess(state: PipelineState) -> dict:
    """Preprocessing node: normalizes the raw input string."""
    raw = state["raw_input"]               # grab raw input from state
    print("[preprocess] cleaning input...") # progress indicator
    cleaned = raw.strip().lower()          # basic normalization
    print("[preprocess] result:", cleaned)  # echo the cleaned result
    return {"preprocessed": cleaned}       # return partial state update


### Node 2: process -- the main computation (optionally with LLM)


In [ ]:
def process(state: PipelineState) -> dict:
    """Processing node: performs the main computation on cleaned input."""
    cleaned = state["preprocessed"]        # get preprocessed data
    print("[process] running main logic...")  # progress indicator

    if LLM_AVAILABLE:                      # use LLM if available
        llm = ChatOllama(model="llama3.1:8b", temperature=0)
        response = llm.invoke([
            HumanMessage(content="Summarize this in one sentence: %s" % cleaned)
        ])
        result = response.content          # extract text from response
    else:                                  # fallback: simple string transform
        result = "Processed: %s (LLM unavailable)" % cleaned

    print("[process] result:", result[:80]) # truncated echo
    return {"processed": result}           # partial state update


### Node 3: postprocess -- format and finalize the output


In [ ]:
def postprocess(state: PipelineState) -> dict:
    """Postprocessing node: adds formatting and final touches."""
    processed = state["processed"]         # get processed data
    print("[postprocess] formatting output...")  # progress indicator
    final = "[FINAL] %s" % processed       # wrap in a marker
    print("[postprocess] done")            # completion signal
    return {"final_output": final}         # return final state update


### Build the graph: preprocess -> process -> postprocess -> END


In [ ]:
builder = StateGraph(PipelineState)        # create builder with our state

builder.add_node("preprocess", preprocess)    # register preprocessing node
builder.add_node("process", process)          # register processing node
builder.add_node("postprocess", postprocess)  # register postprocessing node

builder.set_entry_point("preprocess")         # start at preprocessing
builder.add_edge("preprocess", "process")     # pre -> process
builder.add_edge("process", "postprocess")    # process -> post
builder.add_edge("postprocess", END)          # post -> done

graph = builder.compile()                    # finalize the graph

print("Graph compiled: preprocess -> process -> postprocess -> END")


### Visualize the graph


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Visualization unavailable:", exc)
    print("Nodes:", list(builder.nodes.keys()))


### invoke(): run the full pipeline, see only the end


In [ ]:
print("=== invoke() -- full pipeline ===")
result = graph.invoke({"raw_input": "  LangGraph is a Framework for building Agent Systems  "})
print()
print("Final output:", result["final_output"])


### stream(): see each node as it completes


In [ ]:
print("=== stream() -- node-by-node progress ===")
print()

step_num = 0                               # counter for progress tracking

for step in graph.stream({"raw_input": "  LangGraph enables stateful AI agents  "}):
    step_num += 1                          # increment step counter
    for node_name, node_output in step.items():  # unpack {node: output}
        print("[step %d] node='%s' completed" % (step_num, node_name))
        # Print the partial state returned by this node
        for key, value in node_output.items():
            preview = str(value)[:60]      # truncate long values
            print("   %s = %s" % (key, preview))

print()
print("Total steps observed: %d (matches 3 nodes)" % step_num)


### stream_mode="updates": just the state diffs


In [ ]:
print("=== stream_mode='updates' -- state diffs only ===")
print()

for step in graph.stream(
    {"raw_input": "  Stream modes give different views  "},
    stream_mode="updates",                # only yield state updates, not full output
):
    for node_name, updates in step.items():
        print("node='%s' updated keys: %s" % (node_name, list(updates.keys())))


### stream_mode="values": full state snapshot at each step


In [ ]:
print("=== stream_mode='values' -- full state snapshots ===")
print()

for i, step in enumerate(graph.stream(
    {"raw_input": "  Full state at each step  "},
    stream_mode="values",                 # yield the entire state after each node
)):
    print("[snapshot %d] keys=%s" % (i + 1, list(step.keys())))
    # Show which fields are populated
    for key in ["raw_input", "preprocessed", "processed", "final_output"]:
        val = step.get(key)
        if val is not None:
            print("   %s: %s" % (key, str(val)[:50]))


### Progress tracking pattern: build a progress bar


In [ ]:
print("=== Progress bar pattern ===")
print()

total_nodes = 3                            # known total (3 nodes in our graph)
current = 0                                # progress counter

for step in graph.stream({"raw_input": "  Progress tracking demo  "}):
    current += 1                           # advance counter
    node_name = list(step.keys())[0]      # get the node that just ran
    bar = "#" * current + "." * (total_nodes - current)  # simple ASCII bar
    print("[%s] %d/%d -- %s" % (bar, current, total_nodes, node_name))

print()
print("NOTEBOOK COMPLETE: multi-node streaming demonstrated successfully")
